# NSCI 270 — Statistical Analysis

This notebook examines whether baseline diet quality moderates the relationship between changes in sleep duration and changes in working-memory-related brain activation.

The analysis uses longitudinal ABCD data from sessions 04A and 06A.

In [ ]:
import sys
import site

user_site = site.getusersitepackages()

# Remove personal site-packages from Python's search path
sys.path = [
    p for p in sys.path
    if not p.startswith(user_site)
]

# Disable the user site for this session
site.ENABLE_USER_SITE = False

print("User site:", user_site)
print("User site still in sys.path:", user_site in sys.path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

sns.set_theme(style="whitegrid")

## 1. Load and Verify Data

Load the cleaned longitudinal dataset and check its dimensions, missing values, and key variables before statistical analysis.

In [ ]:
data_path = Path.home() / "NSCI270" / "ABCD_cleaned_longitudinal.csv"

df = pd.read_csv(data_path)

print("Participants:", len(df))
print("Variables:", len(df.columns))
print("Duplicate participants:", df["participant_id"].duplicated().sum())
print("Missing values:", df.isna().sum().sum())

required_cols = [
    "delta_sleep",
    "delta_DQS",
    "DQS_ses-04A",
    "ab_g_dyn__visit_age",
    "ab_g_dyn__design_site",
    "ab_g_stc__cohort_sex",
    "motion_04A",
    "motion_06A",
    "delta_motion"
]

print("\nMissing required columns:")
print(set(required_cols) - set(df.columns))

print("\nROI change variables:")
print([c for c in df.columns if c.startswith("delta_mr_y_tfmri")])

## 2. Descriptive Statistics

### 2.1 Participant Characteristics

Summarize baseline age, sex, and study site distribution in the final longitudinal sample.

In [ ]:
# Baseline age
print("Baseline Age:")
print(
    df["ab_g_dyn__visit_age"]
    .describe()
    .round(2)
)

# Sex distribution
print("\nSex Distribution:")
print(
    df["ab_g_stc__cohort_sex"]
    .value_counts(dropna=False)
)

# Study sites
print("\nNumber of Study Sites:")
print(df["ab_g_dyn__design_site"].nunique())

print("\nParticipants per Site:")
print(
    df["ab_g_dyn__design_site"]
    .value_counts()
    .sort_index()
)

### 2.2 Longitudinal Descriptive Statistics

Summarize sleep duration, diet quality, and frontal brain activation at both sessions. Calculate the mean and standard deviation of longitudinal changes.

In [ ]:
# Variables to summarize
variables = {
    "Sleep (hours)": "ph_y_mctq__sleep_dur",
    "Diet Quality Score": "DQS"
}

for roi in ["cmfrt", "rmfrt", "sfrt"]:
    for hemi in ["lh", "rh"]:
        name = f"{roi.upper()}_{hemi.upper()}"
        col = f"mr_y_tfmri__nback__2bv0b__dsk__{roi}__{hemi}_beta"
        variables[name] = col

# Create summary table
results = []

for name, col in variables.items():
    baseline = df[f"{col}_ses-04A"]
    followup = df[f"{col}_ses-06A"]
    change = followup - baseline

    results.append({
        "Variable": name,
        "Baseline Mean": baseline.mean(),
        "Baseline SD": baseline.std(),
        "Follow-up Mean": followup.mean(),
        "Follow-up SD": followup.std(),
        "Change Mean": change.mean(),
        "Change SD": change.std()
    })

descriptive_table = pd.DataFrame(results)

display(descriptive_table.round(3))

In [ ]:
# Summary statistics from longitudinal analysis
n = 3672

sleep_means = [8.117, 7.847]
sleep_sds = [1.350, 1.221]

t_critical = stats.t.ppf(0.975, df=n - 1)

for session, mean, sd in zip(
    ["ses-04A", "ses-06A"],
    sleep_means,
    sleep_sds
):
    se = sd / np.sqrt(n)
    margin = t_critical * se

    print(
        f"{session}: Mean = {mean:.3f}, "
        f"95% CI = [{mean-margin:.3f}, {mean+margin:.3f}]"
    )

In [ ]:
n = 3672

dqs_means = [0.000, -0.015]
dqs_sds = [0.545, 0.570]

t_critical = stats.t.ppf(0.975, df=n - 1)

for session, mean, sd in zip(
    ["ses-04A", "ses-06A"],
    dqs_means,
    dqs_sds
):
    margin = t_critical * sd / np.sqrt(n)

    print(
        f"{session}: Mean = {mean:.3f}, "
        f"95% CI = [{mean-margin:.3f}, {mean+margin:.3f}], "
        f"Error = {margin:.3f}"
    )

## 3. Paired t-tests

Paired t-tests compare sleep duration, diet quality, and frontal brain activation between baseline and follow-up sessions. These tests determine whether the average longitudinal changes are statistically significant.

In [ ]:
ttest_results = []

for name, col in variables.items():
    baseline = df[f"{col}_ses-04A"]
    followup = df[f"{col}_ses-06A"]

    t_stat, p_value = stats.ttest_rel(followup, baseline)

    ttest_results.append({
        "Variable": name,
        "Mean Change": (followup - baseline).mean(),
        "t": t_stat,
        "p": p_value
    })

ttest_table = pd.DataFrame(ttest_results)

display(ttest_table.round(4))

### 3.1 Multiple Comparison Correction

Apply Benjamini–Hochberg FDR correction across the six frontal ROI paired t-tests to account for multiple comparisons.

In [ ]:
# Select the six ROI tests
roi_mask = ttest_table["Variable"].str.contains(
    "CMFRT|RMFRT|SFRT"
)

roi_tests = ttest_table.loc[roi_mask].copy()

# Apply BH-FDR correction
roi_tests["p_FDR"] = multipletests(
    roi_tests["p"],
    method="fdr_bh"
)[1]

roi_tests["Significant_FDR"] = roi_tests["p_FDR"] < 0.05

display(roi_tests.round(4))

In [ ]:
# Figure 2C: 95% CI for longitudinal changes in frontal ROIs

roi_ci_results = []

for label, col in difference_cols.items():

    # Only include the six frontal ROIs
    if label not in [
        "CMFRT_LH", "CMFRT_RH",
        "RMFRT_LH", "RMFRT_RH",
        "SFRT_LH", "SFRT_RH"
    ]:
        continue

    changes = df[col].dropna()

    n = len(changes)
    mean_change = changes.mean()
    sd_change = changes.std(ddof=1)

    se = sd_change / np.sqrt(n)
    t_critical = stats.t.ppf(0.975, df=n - 1)

    margin = t_critical * se

    roi_ci_results.append({
        "ROI": label,
        "N": n,
        "Mean Change": mean_change,
        "Lower 95% CI": mean_change - margin,
        "Upper 95% CI": mean_change + margin,
        "Error (+/-)": margin
    })

roi_ci_table = pd.DataFrame(roi_ci_results)

display(roi_ci_table.round(5))

In [ ]:
# Display complete paired t-test results for Table 2

print("All paired t-tests:")
display(ttest_table.round(6))

print("ROI tests with FDR correction:")
display(roi_tests.round(6))

### 3.2 Paired t-Test Assumption Checks

Q-Q plots were used to examine the distributions of within-participant differences in sleep duration, Diet Quality Score (DQS), and activation across the six frontal ROIs.

These plots were used to assess approximate normality and identify potential extreme values before interpreting the paired-sample t-test results.

In [ ]:
# Section 3.3: Paired t-test assumption checks

import matplotlib.pyplot as plt
from scipy import stats

# Variables used in the paired t-tests
difference_cols = {
    "Sleep Duration": "delta_sleep",
    "Diet Quality Score": "delta_DQS",
    "CMFRT_LH": "delta_mr_y_tfmri__nback__2bv0b__dsk__cmfrt__lh_beta",
    "CMFRT_RH": "delta_mr_y_tfmri__nback__2bv0b__dsk__cmfrt__rh_beta",
    "RMFRT_LH": "delta_mr_y_tfmri__nback__2bv0b__dsk__rmfrt__lh_beta",
    "RMFRT_RH": "delta_mr_y_tfmri__nback__2bv0b__dsk__rmfrt__rh_beta",
    "SFRT_LH": "delta_mr_y_tfmri__nback__2bv0b__dsk__sfrt__lh_beta",
    "SFRT_RH": "delta_mr_y_tfmri__nback__2bv0b__dsk__sfrt__rh_beta"
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, (label, col) in zip(axes.flat, difference_cols.items()):

    values = df[col].dropna()

    stats.probplot(values, dist="norm", plot=ax)

    ax.set_title(label)
    ax.set_xlabel("Theoretical Quantiles")
    ax.set_ylabel("Observed Differences")

plt.suptitle(
    "Q-Q Plots of Within-Participant Differences",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
# Section 3.2: Outlier screening for paired differences

outlier_results = []

for label, col in difference_cols.items():
    values = df[col].dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    flagged = (values < lower) | (values > upper)

    outlier_results.append({
        "Variable": label,
        "N": len(values),
        "Min": values.min(),
        "Max": values.max(),
        "N_Flagged": flagged.sum(),
        "Percent_Flagged": flagged.mean() * 100
    })

outlier_table = pd.DataFrame(outlier_results)

display(outlier_table.round(3))

## 4. Pearson Correlation Analysis

Examine the linear relationships between changes in sleep duration, baseline diet quality, and changes in frontal brain activation using Pearson correlations.

In [ ]:
# Select variables for correlation analysis
correlation_vars = {
    "Delta Sleep": "delta_sleep",
    "Baseline DQS": "DQS_ses-04A"
}

for roi in ["cmfrt", "rmfrt", "sfrt"]:
    for hemi in ["lh", "rh"]:
        name = f"{roi.upper()}_{hemi.upper()}"
        col = f"delta_mr_y_tfmri__nback__2bv0b__dsk__{roi}__{hemi}_beta"
        correlation_vars[name] = col

# Calculate Pearson correlation matrix
corr_data = df[list(correlation_vars.values())].copy()
corr_data.columns = list(correlation_vars.keys())

corr_matrix = corr_data.corr(method="pearson")

display(corr_matrix.round(3))

### 4.1 Correlation Significance Testing

Calculate Pearson correlation coefficients and p-values between sleep changes, baseline diet quality, and activation changes in six frontal ROIs. Apply FDR correction to the 12 exploratory comparisons.

In [ ]:
correlation_results = []

predictors = ["Delta Sleep", "Baseline DQS"]
rois = ["CMFRT_LH", "CMFRT_RH",
        "RMFRT_LH", "RMFRT_RH",
        "SFRT_LH", "SFRT_RH"]

for predictor in predictors:
    for roi in rois:
        r, p = stats.pearsonr(
            corr_data[predictor],
            corr_data[roi]
        )

        correlation_results.append({
            "Predictor": predictor,
            "ROI": roi,
            "r": r,
            "p": p
        })

correlation_table = pd.DataFrame(correlation_results)

# FDR correction across 12 exploratory tests
correlation_table["p_FDR"] = multipletests(
    correlation_table["p"],
    method="fdr_bh"
)[1]

correlation_table["Significant_FDR"] = (
    correlation_table["p_FDR"] < 0.05
)

display(correlation_table.round(4))

### 4.2 Correlation Heatmap

Visualize Pearson correlations between sleep changes, baseline diet quality, and changes in frontal brain activation.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

sns.heatmap(
    corr_matrix,
    annot=False,       # Disable Seaborn annotations
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    ax=ax
)

# Manually add correlation coefficients to every cell
for i in range(corr_matrix.shape[0]):
    for j in range(corr_matrix.shape[1]):

        value = corr_matrix.iloc[i, j]

        if np.isfinite(value):
            ax.text(
                j + 0.5,
                i + 0.5,
                f"{value:.3f}",
                ha="center",
                va="center",
                fontsize=9,
                color="black"
            )

ax.set_title("Pearson Correlation Matrix")

plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.3 Correlation Assumption Checks

Scatterplots were used to examine the linear relationships between changes in sleep duration and changes in activation across the six frontal ROIs.

Linear trendlines were included to visualize the direction of each association and identify potential outliers.

In [ ]:
# Section 4.2: Correlation Assumption Checks

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

roi_cols = {
    "CMFRT_LH": "delta_mr_y_tfmri__nback__2bv0b__dsk__cmfrt__lh_beta",
    "CMFRT_RH": "delta_mr_y_tfmri__nback__2bv0b__dsk__cmfrt__rh_beta",
    "RMFRT_LH": "delta_mr_y_tfmri__nback__2bv0b__dsk__rmfrt__lh_beta",
    "RMFRT_RH": "delta_mr_y_tfmri__nback__2bv0b__dsk__rmfrt__rh_beta",
    "SFRT_LH": "delta_mr_y_tfmri__nback__2bv0b__dsk__sfrt__lh_beta",
    "SFRT_RH": "delta_mr_y_tfmri__nback__2bv0b__dsk__sfrt__rh_beta"
}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, (roi, col) in zip(axes.flat, roi_cols.items()):

    data = df[["delta_sleep", col]].dropna()

    x = data["delta_sleep"].to_numpy()
    y = data[col].to_numpy()

    # Pearson correlation
    r, p = pearsonr(x, y)

    # Linear trendline
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)

    ax.scatter(x, y, alpha=0.2, s=10)
    ax.plot(x_line, slope * x_line + intercept, color="red")

    ax.set_title(f"{roi} (r = {r:.3f}, p = {p:.3f})")
    ax.set_xlabel("Change in Sleep Duration (hours)")
    ax.set_ylabel("Change in Activation")

plt.suptitle(
    "Longitudinal Sleep–Activation Correlations",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
# Section 4.3: Baseline DQS vs. Activation Change

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, (roi, col) in zip(axes.flat, roi_cols.items()):

    data = df[["DQS_ses-04A", col]].dropna()

    x = data["DQS_ses-04A"].to_numpy()
    y = data[col].to_numpy()

    # Pearson correlation
    r, p = pearsonr(x, y)

    # Linear trendline
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)

    ax.scatter(x, y, alpha=0.2, s=10)
    ax.plot(x_line, slope * x_line + intercept, color="red")

    ax.set_title(f"{roi} (r = {r:.3f}, p = {p:.3f})")
    ax.set_xlabel("Baseline Diet Quality Score")
    ax.set_ylabel("Change in Activation")

plt.suptitle(
    "Baseline Diet Quality and Longitudinal Activation Changes",
    fontsize=15
)

plt.tight_layout()
plt.show()

### 4.4 Anatomical Map of Sleep–Activation Correlations

### 4.5 Correlation Coefficient 

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

sleep_col = "delta_sleep"
diet_col = "DQS_ses-04A"

roi_cols = {
    "CMFRT_LH": "delta_mr_y_tfmri__nback__2bv0b__dsk__cmfrt__lh_beta",
    "CMFRT_RH": "delta_mr_y_tfmri__nback__2bv0b__dsk__cmfrt__rh_beta",
    "RMFRT_LH": "delta_mr_y_tfmri__nback__2bv0b__dsk__rmfrt__lh_beta",
    "RMFRT_RH": "delta_mr_y_tfmri__nback__2bv0b__dsk__rmfrt__rh_beta",
    "SFRT_LH": "delta_mr_y_tfmri__nback__2bv0b__dsk__sfrt__lh_beta",
    "SFRT_RH": "delta_mr_y_tfmri__nback__2bv0b__dsk__sfrt__rh_beta"
}

results = []

for predictor, column in {
    "Delta Sleep": sleep_col,
    "Baseline DQS": diet_col
}.items():

    for roi, roi_column in roi_cols.items():

        paired = df[[column, roi_column]].dropna()
        n = len(paired)

        r, p = pearsonr(paired[column], paired[roi_column])

        z = np.arctanh(r)
        se = 1 / np.sqrt(n - 3)

        ci_low = np.tanh(z - 1.96 * se)
        ci_high = np.tanh(z + 1.96 * se)

        results.append({
            "Predictor": predictor,
            "ROI": roi,
            "N": n,
            "r": r,
            "CI_low": ci_low,
            "CI_high": ci_high,
            "p": p
        })

corr_plot_df = pd.DataFrame(results)

display(corr_plot_df.round(5))

## 5. Moderation Regression

### 5.1 Prepare Regression Variables

Test whether baseline diet quality moderates the association between changes in sleep duration and changes in frontal brain activation. Models will adjust for age, sex, study site, and N-back head motion.

In [ ]:
# Create a dataset with shorter variable names
reg_df = df.copy()

reg_df["sleep_change"] = reg_df["delta_sleep"]
reg_df["diet_baseline"] = reg_df["DQS_ses-04A"]
reg_df["age"] = reg_df["ab_g_dyn__visit_age"]

reg_df["sex"] = reg_df["ab_g_stc__cohort_sex"].astype("category")
reg_df["site"] = reg_df["ab_g_dyn__design_site"].astype("category")

# Center predictors for easier interpretation
reg_df["sleep_c"] = (
    reg_df["sleep_change"] - reg_df["sleep_change"].mean()
)

reg_df["diet_c"] = (
    reg_df["diet_baseline"] - reg_df["diet_baseline"].mean()
)

# Interaction term
reg_df["sleep_diet"] = reg_df["sleep_c"] * reg_df["diet_c"]

# Verify variables
print("Participants:", len(reg_df))

print("\nRegression variables:")
print(
    reg_df[
        ["sleep_c", "diet_c", "sleep_diet",
         "age", "motion_04A", "delta_motion"]
    ].describe().round(3)
)

print("\nSex categories:", reg_df["sex"].cat.categories.tolist())
print("Number of sites:", reg_df["site"].nunique())

### 5.2 Moderation Regression Models

Fit separate linear regression models for six frontal ROIs. Each model includes the Sleep × Diet interaction and adjusts for baseline age, sex, study site, baseline head motion, and change in head motion.

In [ ]:
regression_results = []
models = {}

for roi in rois:

    roi_type, hemi = roi.lower().split("_")

    outcome = (
        "delta_mr_y_tfmri__nback__2bv0b__dsk__"
        f"{roi_type}__{hemi}_beta"
    )

    formula = (
        f"{outcome} ~ sleep_c * diet_c + age "
        "+ C(sex) + C(site) + motion_04A + delta_motion"
    )

    model = smf.ols(formula, data=reg_df).fit()

    models[roi] = model

    interaction = "sleep_c:diet_c"

    regression_results.append({
        "ROI": roi,
        "Beta": model.params[interaction],
        "SE": model.bse[interaction],
        "t": model.tvalues[interaction],
        "p": model.pvalues[interaction],
        "CI_Lower": model.conf_int().loc[interaction, 0],
        "CI_Upper": model.conf_int().loc[interaction, 1],
        "R_squared": model.rsquared
    })

regression_table = pd.DataFrame(regression_results)

display(regression_table.round(4))

### 5.3 Multiple Comparison Correction

Apply Benjamini–Hochberg FDR correction across the six primary Sleep × Diet interaction tests.

In [ ]:
# Correct the six interaction p-values
regression_table["p_FDR"] = multipletests(
    regression_table["p"],
    method="fdr_bh"
)[1]

regression_table["Significant_FDR"] = (
    regression_table["p_FDR"] < 0.05
)

display(regression_table.round(4))

In [ ]:
from statsmodels.stats.multitest import multipletests
# Data used to generate table and graphs
interaction = "sleep_c:diet_c"
results = []

for roi, model in models.items():

    ci = model.conf_int().loc[interaction]

    results.append({
        "ROI": roi,
        "β": model.params[interaction],
        "95% CI Lower": ci.iloc[0],
        "95% CI Upper": ci.iloc[1],
        "p-value": model.pvalues[interaction],
        "R²": model.rsquared
    })

table3 = pd.DataFrame(results)

# FDR correction across the six interaction tests
table3["FDR-adjusted p"] = multipletests(
    table3["p-value"],
    method="fdr_bh"
)[1]

# Display results
display(table3.round(5))

# Save summary table
table3.to_csv("table3_moderation_results.csv", index=False)

### 5.4 Regression Model Diagnostics

Examine the residual distributions and residual-versus-fitted plots to assess the assumptions of the OLS regression models.

The Q-Q plot is used to evaluate residual normality, while the residuals-versus-fitted plot is used to identify potential nonlinearity, heteroscedasticity, and outliers.

In [ ]:
# Section 5.4: Regression Model Diagnostics

# Select one of the fitted moderation models
model = models["CMFRT_LH"]

# Extract residuals and fitted values
residuals = model.resid
fitted = model.fittedvalues

# Create diagnostic plots
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Q-Q plot: residual normality
stats.probplot(
    residuals,
    dist="norm",
    plot=axes[0]
)
axes[0].set_title("Q-Q Plot: CMFRT_LH")

# Residuals vs. fitted values
axes[1].scatter(
    fitted,
    residuals,
    alpha=0.3,
    s=10
)

axes[1].axhline(
    0,
    color="red",
    linestyle="--"
)

axes[1].set_xlabel("Fitted Values")
axes[1].set_ylabel("Residuals")
axes[1].set_title("Residuals vs. Fitted: CMFRT_LH")

plt.tight_layout()
plt.show()

In [ ]:
# Section 5.4: Diagnostics for the remaining five ROIs

remaining_rois = [
    "CMFRT_RH",
    "RMFRT_LH",
    "RMFRT_RH",
    "SFRT_LH",
    "SFRT_RH"
]

for roi in remaining_rois:
    model = models[roi]

    residuals = model.resid
    fitted = model.fittedvalues

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # Q-Q plot
    stats.probplot(residuals, dist="norm", plot=axes[0])
    axes[0].set_title(f"Q-Q Plot: {roi}")

    # Residuals vs. fitted
    axes[1].scatter(fitted, residuals, alpha=0.3, s=10)
    axes[1].axhline(0, color="red", linestyle="--")
    axes[1].set_xlabel("Fitted Values")
    axes[1].set_ylabel("Residuals")
    axes[1].set_title(f"Residuals vs. Fitted: {roi}")

    plt.tight_layout()
    plt.show()

### 5.5 Multicollinearity Assessment

Variance inflation factors (VIFs) were calculated to evaluate multicollinearity among predictors in the multiple linear regression models.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# All six models use the same predictors
model = models["CMFRT_LH"]

X = model.model.exog
predictor_names = model.model.exog_names

vif_table = pd.DataFrame({
    "Predictor": predictor_names,
    "VIF": [
        variance_inflation_factor(X, i)
        for i in range(X.shape[1])
    ]
})

# Exclude intercept
vif_table = vif_table[
    vif_table["Predictor"] != "Intercept"
]

display(vif_table.round(3))

### 5.6 Influential Observations

Cook's distance was calculated to identify potentially influential observations in the six regression models. Observations exceeding the conventional screening threshold of 4/N were flagged for further examination rather than automatically excluded.

In [ ]:
import numpy as np
import pandas as pd

influence_results = []

for roi, model in models.items():

    # Calculate Cook's distance
    cooks_d = model.get_influence().cooks_distance[0]

    n = int(model.nobs)
    threshold = 4 / n

    influence_results.append({
        "ROI": roi,
        "N": n,
        "Threshold_4_over_N": threshold,
        "Max_Cooks_D": np.max(cooks_d),
        "N_Flagged": np.sum(cooks_d > threshold),
        "Percent_Flagged": 100 * np.mean(cooks_d > threshold),
        "N_Above_1": np.sum(cooks_d > 1)
    })

influence_table = pd.DataFrame(influence_results)

display(influence_table.round(5))

### 5.7 Sensitivity Analysis: HC3 Robust Standard Errors

HC3 heteroscedasticity-robust standard errors were used as a sensitivity analysis to evaluate whether the statistical inference for the Sleep × Diet Quality Score interaction was sensitive to potential heteroscedasticity.

The original regression coefficients were retained, while standard errors, confidence intervals, and p-values were recalculated using HC3. Benjamini–Hochberg FDR correction was applied across the six primary interaction tests.

In [ ]:
# Section 5.7: HC3 Robust Standard Errors

import pandas as pd
from statsmodels.stats.multitest import multipletests

hc3_results = []

for roi, model in models.items():

    # Recalculate standard errors using HC3
    robust = model.get_robustcov_results(cov_type="HC3")

    # Locate the Sleep x Diet interaction
    interaction = "sleep_c:diet_c"
    idx = model.model.exog_names.index(interaction)

    hc3_results.append({
        "ROI": roi,
        "Beta": model.params[interaction],
        "SE_OLS": model.bse[interaction],
        "SE_HC3": robust.bse[idx],
        "p_OLS": model.pvalues[interaction],
        "p_HC3": robust.pvalues[idx],
        "CI_Lower_HC3": robust.conf_int()[idx, 0],
        "CI_Upper_HC3": robust.conf_int()[idx, 1]
    })

hc3_table = pd.DataFrame(hc3_results)

# FDR correction across six interaction tests
hc3_table["p_FDR_HC3"] = multipletests(
    hc3_table["p_HC3"],
    method="fdr_bh"
)[1]

hc3_table["Significant_FDR_HC3"] = (
    hc3_table["p_FDR_HC3"] < 0.05
)

display(hc3_table.round(5))

### 5.8 Influence Sensitivity Analysis

A sensitivity analysis was conducted to evaluate whether potentially influential observations affected the estimated Sleep × DQS interaction coefficients.

For each ROI, observations with Cook's distance greater than 4/N were temporarily excluded, and the original regression model was refitted using the remaining observations. Regression coefficients and p-values were compared with the full-sample models.

This analysis was exploratory. The original dataset and primary regression results were retained without excluding participants.

In [ ]:
# Section 5.8: Influence Sensitivity Analysis

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

sensitivity_results = []

for roi, model in models.items():

    # Original full-sample model
    interaction = "sleep_c:diet_c"
    n_original = int(model.nobs)

    # Identify potentially influential observations
    cooks_d = model.get_influence().cooks_distance[0]
    threshold = 4 / n_original
    keep = np.isfinite(cooks_d) & (cooks_d <= threshold)

    # Use the exact observations and formula from the original model
    original_data = model.model.data.frame.loc[
        model.model.data.row_labels
    ].copy()

    filtered_data = original_data.iloc[
        np.flatnonzero(keep)
    ].copy()

    # Refit the same regression model
    sensitivity_model = smf.ols(
        formula=model.model.formula,
        data=filtered_data
    ).fit()

    sensitivity_results.append({
        "ROI": roi,
        "N_Original": n_original,
        "N_Excluded": int((~keep).sum()),
        "N_Remaining": int(keep.sum()),
        "Beta_Original": model.params[interaction],
        "Beta_Sensitivity": sensitivity_model.params[interaction],
        "p_Original": model.pvalues[interaction],
        "p_Sensitivity": sensitivity_model.pvalues[interaction]
    })

sensitivity_table = pd.DataFrame(sensitivity_results)

# FDR correction across the six sensitivity models
sensitivity_table["p_FDR_Sensitivity"] = multipletests(
    sensitivity_table["p_Sensitivity"],
    method="fdr_bh"
)[1]

display(sensitivity_table.round(5))